# Step 2 — Fine-tune MedGemma 4B to write chest X-ray reports (IU-Xray)

This notebook teaches **MedGemma 4B** (Google's open medical vision-language model) to write the *Findings* and *Impression* of a chest X-ray report in the IU-Xray style.

**Method: QLoRA.** The model is loaded in 4-bit to fit on a free GPU, and only small LoRA adapter layers (~1% of the weights) are trained. The output is a small adapter folder you add on top of the original model.

**What you get at the end**
- A loss curve showing the model learning
- A **before vs after** comparison on test images (BLEU-4 and ROUGE-L), plus side-by-side reports you can read
- The trained adapter, saved and zipped for download

## Before you start (one-time setup)

1. **GPU**
   - **Kaggle:** *Settings → Accelerator → GPU T4 x2*, and *Settings → Internet → On*. Also add the dataset `raddar/chest-xrays-indiana-university` (*Add Input*).
   - **Colab:** *Runtime → Change runtime type → T4 GPU* (free). An L4 or A100 (Colab Pro) is much faster.
2. **Model access:** MedGemma is free but gated.
   - Create a Hugging Face account.
   - Open https://huggingface.co/google/medgemma-4b-it and accept the terms.
   - Create a **Read** token at https://huggingface.co/settings/tokens.
3. **Add the token as a secret named `HF_TOKEN`**
   - **Kaggle:** *Add-ons → Secrets*
   - **Colab:** the 🔑 icon in the left sidebar, with notebook access switched on

**Do a quick run first.** `QUICK_RUN = True` trains on 300 images, enough to check that everything works end to end. Then set it to `False` for the real training. A full epoch on a free T4 can take several hours. On Kaggle, use *Save Version → Save & Run All* so it keeps running in the background.

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate kagglehub sacrebleu rouge-score

## 1. Settings

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU (Kaggle gives two; the 4-bit model needs just one)

MODEL_ID       = "google/medgemma-4b-it"
QUICK_RUN      = True     # True = 300 training images to test the pipeline. Set False for the full training.
NUM_EPOCHS     = 1
LEARNING_RATE  = 2e-4
GRAD_ACCUM     = 8        # effective batch size = 1 x GRAD_ACCUM
MAX_NEW_TOKENS = 200      # longest report the model may write (IU-Xray reports are short)
N_TEST_EVAL    = 50       # test images used for the before/after comparison
OUTPUT_DIR     = "medgemma-cxr-lora"
SEED           = 42

PROMPT = ("You are an expert chest radiologist. Write the FINDINGS and IMPRESSION "
          "sections of the radiology report for this chest X-ray.")

## 2. Log in to Hugging Face

In [ ]:
from huggingface_hub import login

def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:                                    # Kaggle secret
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:                                    # Colab secret
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    from getpass import getpass
    return getpass("Paste your Hugging Face token: ")

login(get_hf_token())

## 3. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found: switch on a GPU accelerator first (see the setup section)."

major, _ = torch.cuda.get_device_capability(0)
BF16 = major >= 8          # L4 / A100 / H100 support bfloat16; T4 and P100 do not
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name(0)}  |  "
      f"memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB  |  compute dtype: {COMPUTE_DTYPE}")

## 4. Prepare the data
Same cleaning as Step 1:
- keep one frontal image per study
- drop studies with no Findings and no Impression
- remove the `XXXX` anonymization token

The target text the model learns to write is `FINDINGS: ...` followed by `IMPRESSION: ...`.

The data is **split by study** (80 / 10 / 10), so the same patient never appears in both training and test.

In [ ]:
import re, random, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

def locate_dataset():
    if os.environ.get("IUXRAY_DIR"):
        return Path(os.environ["IUXRAY_DIR"])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        hits = list(kaggle_input.rglob("indiana_reports.csv"))
        if hits:
            return hits[0].parent
    import kagglehub
    return Path(kagglehub.dataset_download("raddar/chest-xrays-indiana-university"))

DATA_DIR = locate_dataset()
IMG_DIR = next(DATA_DIR.rglob("*.png")).parent
reports = pd.read_csv(next(DATA_DIR.rglob("indiana_reports.csv")))
proj = pd.read_csv(next(DATA_DIR.rglob("indiana_projections.csv")))

def clean(t):
    t = "" if pd.isna(t) else str(t)
    t = re.sub(r"\bXXXX\b", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+([.,;])", r"\1", t)
    t = re.sub(r"([.,;])(\s*[.,;])+", r"\1", t)
    t = re.sub(r"\s{2,}", " ", t).strip()
    return t[:1].upper() + t[1:]

def make_target(findings, impression):
    parts = []
    if findings:
        parts.append(f"FINDINGS: {findings}")
    if impression:
        parts.append(f"IMPRESSION: {impression}")
    return "\n".join(parts)

frontal = proj[proj["projection"] == "Frontal"].drop_duplicates("uid").set_index("uid")["filename"]
df = reports[reports["uid"].isin(frontal.index)].copy()
df["filename"] = df["uid"].map(frontal)
df["findings"] = df["findings"].map(clean)
df["impression"] = df["impression"].map(clean)
df["target"] = [make_target(f, i) for f, i in zip(df["findings"], df["impression"])]
df = df[df["target"] != ""].reset_index(drop=True)

# split by study id
uids = df["uid"].unique().tolist()
random.Random(SEED).shuffle(uids)
n = len(uids)
split_of = {u: "train" for u in uids[: int(0.8 * n)]}
split_of.update({u: "val" for u in uids[int(0.8 * n): int(0.9 * n)]})
split_of.update({u: "test" for u in uids[int(0.9 * n):]})
df["split"] = df["uid"].map(split_of)

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"].head(100)          # a small validation set keeps evaluation fast
test_df = df[df["split"] == "test"]
if QUICK_RUN:
    train_df = train_df.sample(min(300, len(train_df)), random_state=SEED)
    val_df = val_df.head(30)
eval_df = test_df.sample(min(N_TEST_EVAL, len(test_df)), random_state=SEED).reset_index(drop=True)

print(f"usable studies: {len(df):,}  ->  train {len(train_df):,} | val {len(val_df):,} | "
      f"test {len(test_df):,} (before/after on {len(eval_df)})")
df.to_csv("iu_xray_splits.csv", index=False)
print("\nExample target text:\n" + train_df.iloc[0]["target"])

## 5. Load MedGemma 4B in 4-bit
This downloads about 8 GB the first time. If you get a 401 or 403 error, the terms on the model page haven't been accepted yet, or the token is wrong.

In [ ]:
import transformers
from packaging import version
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
dtype_arg = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="eager",       # recommended for training Gemma 3 models
    device_map={"": 0},
    **{dtype_arg: COMPUTE_DTYPE},
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"
print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6. Report-writing function, and the zero-shot baseline ("before")
Before training, we ask the original model to write reports for the evaluation images. This is the *before* in the before/after comparison.

In [ ]:
from tqdm.auto import tqdm

def load_image(filename):
    return Image.open(IMG_DIR / filename).convert("RGB")

def build_messages(target=None):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    if target is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return msgs

@torch.inference_mode()
def generate_reports(filenames, batch_size=4):
    model.eval()
    processor.tokenizer.padding_side = "left"           # left padding for batched generation
    prompt = processor.apply_chat_template(build_messages(), add_generation_prompt=True, tokenize=False)
    outputs = []
    for i in tqdm(range(0, len(filenames), batch_size), desc="writing reports"):
        batch = list(filenames[i:i + batch_size])
        inputs = processor(text=[prompt] * len(batch), images=[[load_image(f)] for f in batch],
                           return_tensors="pt", padding=True).to(model.device)
        inputs["pixel_values"] = inputs["pixel_values"].to(COMPUTE_DTYPE)
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs += [t.strip() for t in processor.batch_decode(new_tokens, skip_special_tokens=True)]
    processor.tokenizer.padding_side = "right"
    return outputs

eval_df["before"] = generate_reports(eval_df["filename"].tolist())
for _, r in eval_df.head(2).iterrows():
    print(f"=== uid {r['uid']} ===\nREFERENCE:\n{r['target']}\n\nBEFORE TRAINING:\n{r['before']}\n")

## 7. Add LoRA adapters
Only the attention and MLP layers of the **language** part get small trainable adapters. The vision encoder stays frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Batch builder (collator)
For each example it builds `[image + prompt] → [report]`.

The loss is computed **only on the report tokens**. The image and prompt tokens are masked with `-100`, so the model learns to write the report, not to repeat the prompt.

In [ ]:
class ReportCollator:
    def __init__(self, processor):
        self.processor = processor
        self.prompt_len = None

    def _prompt_length(self, image):
        prompt = self.processor.apply_chat_template(build_messages(), add_generation_prompt=True, tokenize=False)
        return self.processor(text=[prompt], images=[[image]], return_tensors="pt")["input_ids"].shape[1]

    def __call__(self, examples):
        images = [load_image(e["filename"]) for e in examples]
        texts = [self.processor.apply_chat_template(build_messages(e["target"]), tokenize=False).strip()
                 for e in examples]
        batch = self.processor(text=texts, images=[[im] for im in images], return_tensors="pt", padding=True)
        if self.prompt_len is None:     # the prompt is identical for every example
            self.prompt_len = self._prompt_length(images[0])
        labels = batch["input_ids"].clone()
        labels[:, : self.prompt_len] = -100
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch

collator = ReportCollator(processor)
train_records = train_df[["filename", "target"]].to_dict("records")
val_records = val_df[["filename", "target"]].to_dict("records")

# sanity check: the part the model is trained to write should be exactly the report
b = collator(train_records[:1])
learned = b["labels"][0][b["labels"][0] != -100]
print(f"sequence length: {b['input_ids'].shape[1]} tokens (prompt part: {collator.prompt_len})")
print("The model learns to write:\n" + processor.decode(learned, skip_special_tokens=False))

## 9. Train

In [ ]:
import math
from transformers import TrainingArguments, Trainer

steps_per_epoch = math.ceil(len(train_records) / GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
eval_every = max(10, total_steps // 5)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * total_steps)),
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    bf16=BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train_records,
                  eval_dataset=val_records, data_collator=collator)
print(f"{len(train_records)} training examples -> {total_steps} optimizer steps")
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
hist = pd.DataFrame(trainer.state.log_history)
plt.figure(figsize=(8, 4))
if "loss" in hist:
    tr = hist.dropna(subset=["loss"])
    plt.plot(tr["step"], tr["loss"], label="training loss")
if "eval_loss" in hist:
    ev = hist.dropna(subset=["eval_loss"])
    plt.plot(ev["step"], ev["eval_loss"], "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Training progress"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("training_loss.png", bbox_inches="tight", dpi=150); plt.show()

## 10. Save the trained adapter
The adapter is small (tens of MB). The zip file appears in the output files on Kaggle, or in the 📁 panel on Colab.

In [ ]:
import shutil
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
shutil.make_archive(OUTPUT_DIR, "zip", final_dir)
print(f"Saved adapter to {final_dir}/ and {OUTPUT_DIR}.zip")

## 11. Before vs after
Same evaluation images, now with the fine-tuned model.
- **BLEU-4** and **ROUGE-L** measure word overlap with the radiologist's report. Higher is better, 0–100.
- These scores don't check medical correctness. Clinical metrics (CheXbert F1, RadGraph F1) come in the next step, so also *read* the examples below.

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer

model.config.use_cache = True
eval_df["after"] = generate_reports(eval_df["filename"].tolist())

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
def text_scores(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    rouge_l = np.mean([scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]) * 100
    return round(bleu, 1), round(rouge_l, 1)

refs = eval_df["target"].tolist()
results = pd.DataFrame([text_scores(eval_df["before"].tolist(), refs),
                        text_scores(eval_df["after"].tolist(), refs)],
                       index=["Before (zero-shot)", "After (fine-tuned)"], columns=["BLEU-4", "ROUGE-L"])
display(results)
eval_df[["uid", "filename", "target", "before", "after"]].to_csv("test_predictions.csv", index=False)

In [ ]:
wrap = lambda s: "\n".join(textwrap.fill(line, 60) for line in str(s).splitlines()) or "—"
show = eval_df.sample(min(3, len(eval_df)), random_state=1)   # change random_state to see others
fig, axes = plt.subplots(len(show), 4, figsize=(20, 5.5 * len(show)),
                         gridspec_kw={"width_ratios": [1, 1.3, 1.3, 1.3]}, squeeze=False)
for row, (_, r) in zip(axes, show.iterrows()):
    row[0].imshow(load_image(r["filename"]), cmap="gray"); row[0].set_title(f"uid {r['uid']}")
    for ax, title, txt in zip(row[1:], ["RADIOLOGIST", "BEFORE TRAINING", "AFTER TRAINING"],
                              [r["target"], r["before"], r["after"]]):
        ax.text(0, 1, f"{title}\n\n{wrap(txt)}", va="top", fontsize=9, family="monospace")
    for ax in row:
        ax.axis("off")
plt.tight_layout(); plt.savefig("before_after_examples.png", bbox_inches="tight", dpi=130); plt.show()

## 12. Using your trained model later
In a new session, load the original model and add your adapter on top:

```python
from peft import PeftModel
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                                   attn_implementation="eager", device_map={"": 0})
model = PeftModel.from_pretrained(base, "medgemma-cxr-lora/final")
processor = AutoProcessor.from_pretrained("medgemma-cxr-lora/final")
```
Then reuse `generate_reports([...filenames...])` from section 6.

## If something goes wrong
| Problem | Fix |
|---|---|
| `401` / `403` / "gated repo" | Accept the terms on the MedGemma model page and check the `HF_TOKEN` secret |
| `CUDA out of memory` | Restart the session and run from the top. Keep batch size 1. Close other notebooks using the GPU |
| Download fails on Kaggle | *Settings → Internet → On* |
| Training is very slow | Normal on a T4 (no bfloat16). Use `QUICK_RUN` to test, then run the full training as a Kaggle background version, or on an L4/A100 |
| "After" reports look worse | Try 2–3 epochs, or lower `LEARNING_RATE` to `1e-4`, and compare validation loss |